# PG-Stack · Step 6 — Revision analyses for the reviewers

| Section | Reviewer request |
|---|---|
| R1 | Cohort table per centre: patients, benign/malignant, prevalence, cancers per patient |
| R2 | One consistent definition for every number (point estimate = mean over repeats; CI = patient bootstrap, B = 1000, fast weighted implementation) |
| R3 | Sensitivity to the constant *k* ∈ {1, 2, 3, 6, 10, 15, 20, 30} (C1 and C2, no selection) |
| R4 | Ablation of the zero-label score *U* (centroid distance, kNN distance, LOF, Mahalanobis, isolation forest, combinations) |
| R5 | Clinical metrics: average precision, precision/recall@5 and @10 per patient, false positives per patient, sensitivity at fixed referral workload |
| R6 | Paired patient-level permutation tests for the prespecified primary hypotheses (+ Holm) |
| R7 | Deployment-style ranking: (a) within-patient ranks, (b) reference-ECDF ranks that can score a single new patient |
| R8 | Subgroups by number of lesions per patient |
| R9 | Leakage audit: nested vs flat stacking; zero-label score with preprocessing fitted on training patients only |

**Needs in `/kaggle/working` (or an attached dataset):** `oof_step3.csv`, `oof_step4c.csv`, `emb_pca32.npy`, `emb_dinov2_f16.npy`, `pgstack_eval.py`, `step5b_lowlabel.npz`, `step5c_c1.npz`, `step5c_c2.npz`.
CPU is enough. Runtime ≈ 60–90 min. Every section saves its CSV as soon as it finishes.

In [ ]:
import os, sys, glob, time, gc, warnings, json
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
COMP_DIR = "/kaggle/input/competitions/isic-2024-challenge"
SEED, SEED_C = 42, 2026
N_BOOT = 1000            # bootstrap resamples for single-run comparisons
N_BOOT_REP = 500         # bootstrap resamples for 10-repeat designs (CIs only; p-values come from R6)
N_PERM = 200             # permutations per primary test (resolution ~0.005)
RUN_NESTED = True        # R9 nested stacking audit (~10 min)
RUN_REFERENCE = True     # R7b reference-ECDF re-run (~10 min)
K_GRID = [1, 2, 3, 6, 10, 15, 20, 30]
FRAC_C1 = [0.01, 0.02, 0.03, 0.05, 0.10, 0.20]; REP_C1 = 10
FRAC_C2 = [0.01, 0.02, 0.05, 0.10]; REP_C2 = 5
FRAC_5B = [0.01, 0.02, 0.05, 0.10]; REP_5B = 10
N_BAGS, NEG_PER_POS, MIN_POS_HOSPITAL = 3, 20, 10

def locate(name):
    if os.path.exists(f"{WORK}/{name}"): return f"{WORK}/{name}"
    for d in range(1, 5):
        hits = glob.glob("/kaggle/input/" + "*/" * d + name)
        if hits: return hits[0]
    return None
NEED = ["oof_step3.csv", "oof_step4c.csv", "emb_pca32.npy", "emb_dinov2_f16.npy", "pgstack_eval.py",
        "step5b_lowlabel.npz", "step5c_c1.npz", "step5c_c2.npz"]
paths = {n: locate(n) for n in NEED}
print(json.dumps(paths, indent=1))
assert all(paths.values()), f"missing: {[k for k, v in paths.items() if v is None]}"
sys.path.insert(0, os.path.dirname(paths["pgstack_eval.py"]))
from pgstack_eval import *
OUT = lambda f: f"{WORK}/rev_{f}"

## Features and zero-label score (identical code to Steps 5–5c, so *U* is reproduced exactly)

In [ ]:
oof = pd.read_csv(paths["oof_step3.csv"])
df = pd.read_csv(f"{COMP_DIR}/train-metadata.csv", low_memory=False)
df = oof[["isic_id", "fold"]].merge(df, on="isic_id", how="left")
assert (df.isic_id.values == oof.isic_id.values).all()
emb = np.load(paths["emb_pca32.npy"]).astype(np.float32)
assert len(emb) == len(oof)

LEAK = ["lesion_id", "iddx_full", "iddx_1", "iddx_2", "iddx_3", "iddx_4", "iddx_5",
        "mel_mitotic_index", "mel_thick_mm", "tbp_lv_dnn_lesion_confidence"]
NUM_RAW = [c for c in df.columns if (c.startswith("tbp_lv_") or c in ["age_approx", "clin_size_long_diam_mm"])
           and c not in LEAK and pd.api.types.is_numeric_dtype(df[c])]
CAT_COLS = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple"]

def add_lesion_features(d):
    d = d.copy(); eps = 1e-6
    d["age_approx"] = pd.to_numeric(d["age_approx"], errors="coerce")
    d["lesion_size_ratio"] = d.tbp_lv_minorAxisMM / (d.clin_size_long_diam_mm + eps)
    d["lesion_shape_index"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM ** 2 + eps)
    d["hue_contrast"] = (d.tbp_lv_H - d.tbp_lv_Hext).abs()
    d["luminance_contrast"] = (d.tbp_lv_L - d.tbp_lv_Lext).abs()
    d["lesion_color_difference"] = np.sqrt(d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2)
    d["border_complexity"] = d.tbp_lv_norm_border + d.tbp_lv_symm_2axis
    d["color_uniformity"] = d.tbp_lv_color_std_mean / (d.tbp_lv_radial_color_std_max + eps)
    d["position_distance_3d"] = np.sqrt(d.tbp_lv_x**2 + d.tbp_lv_y**2 + d.tbp_lv_z**2)
    d["perimeter_to_area_ratio"] = d.tbp_lv_perimeterMM / (d.tbp_lv_areaMM2 + eps)
    d["area_to_perimeter_ratio"] = d.tbp_lv_areaMM2 / (d.tbp_lv_perimeterMM + eps)
    d["lesion_visibility_score"] = d.tbp_lv_deltaLBnorm + d.tbp_lv_norm_color
    d["symmetry_border_consistency"] = d.tbp_lv_symm_2axis * d.tbp_lv_norm_border
    d["color_shape_composite"] = (d.tbp_lv_color_std_mean + d.tbp_lv_area_perim_ratio + d.tbp_lv_symm_2axis) / 3
    d["log_area"] = np.log1p(d.tbp_lv_areaMM2)
    d["normalized_lesion_size"] = d.clin_size_long_diam_mm / (d.age_approx + eps)
    d["mean_hue_difference"] = (d.tbp_lv_H + d.tbp_lv_Hext) / 2
    d["std_dev_contrast"] = np.sqrt((d.tbp_lv_deltaA**2 + d.tbp_lv_deltaB**2 + d.tbp_lv_deltaL**2) / 3)
    return d

ENG = ["lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast", "lesion_color_difference",
       "border_complexity", "color_uniformity", "position_distance_3d", "perimeter_to_area_ratio",
       "area_to_perimeter_ratio", "lesion_visibility_score", "symmetry_border_consistency",
       "color_shape_composite", "log_area", "normalized_lesion_size", "mean_hue_difference", "std_dev_contrast"]
df = add_lesion_features(df)
LESION_NUM = sorted(set(NUM_RAW + ENG))
for c in CAT_COLS: df[c] = df[c].fillna("NA").astype("category")
y, folds, pid = oof.target.values, oof.fold.values, oof.patient_id.values
N_FOLDS = int(folds.max()) + 1
POS_COLS = ["tbp_lv_x", "tbp_lv_y", "tbp_lv_z", "position_distance_3d"]
TAB_COLS = [c for c in LESION_NUM if c not in POS_COLS]        # appearance/shape only, for the tabular graph
print(len(df), "lesions |", len(LESION_NUM), "lesion features")

In [ ]:
df["hospital"] = df["attribution"].fillna("unknown").astype(str)
print(df.groupby("hospital").agg(lesions=("isic_id", "size"), cancers=("target", "sum"),
                                 patients=("patient_id", "nunique")).sort_values("cancers", ascending=False))

In [ ]:
g = df.groupby("patient_id"); GLOBAL = []
for c in LESION_NUM:
    df[f"{c}_pnorm"] = (df[c] - g[c].transform("mean")) / (g[c].transform("std") + 1e-6); GLOBAL.append(f"{c}_pnorm")
df["patient_n_lesions"] = g["isic_id"].transform("size"); GLOBAL.append("patient_n_lesions")

In [ ]:
X = df[LESION_NUM].astype(np.float32); X = X.fillna(X.median()).values
tab_idx = [LESION_NUM.index(c) for c in TAB_COLS]
TAB = StandardScaler().fit_transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
TAB10 = PCA(10, random_state=SEED).fit_transform(TAB).astype(np.float32)
XYZ = df[["tbp_lv_x", "tbp_lv_y", "tbp_lv_z"]].fillna(0).values.astype(np.float32)
SP = {"tab": TAB, "app": emb, "pos": XYZ}
n = len(df)
GP = {f"gp_{s}_k{k}_{t}": np.full(n, np.nan, np.float32) for s in SP for k in [5, 10, 20] for t in ["dist", "lof"]}
GP["gp_mahal"] = np.full(n, np.nan, np.float32); GP["gp_iforest"] = np.full(n, np.nan, np.float32)

def lof(dist, idx, k):
    d, i = dist[:, :k], idx[:, :k]; reach = np.maximum(d, d[:, -1][i])
    lrd = 1.0 / (reach.mean(1) + 1e-6); return lrd[i].mean(1) / lrd

t0 = time.time()
for j, (p, rows) in enumerate(pd.Series(np.arange(n)).groupby(pid).indices.items()):
    r = np.asarray(rows); m = len(r)
    if m < 6: continue
    km = min(20, m - 1)
    for s, Z in SP.items():
        dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(Z[r]).kneighbors(Z[r]); dist, idx = dist[:, 1:], idx[:, 1:]
        for k in [5, 10, 20]:
            kk = min(k, km); GP[f"gp_{s}_k{k}_dist"][r] = dist[:, :kk].mean(1); GP[f"gp_{s}_k{k}_lof"][r] = lof(dist, idx, kk)
    if m >= 15:
        Zp = TAB10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
        GP["gp_mahal"][r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
    if m >= 10:
        Zi = np.c_[TAB10[r], emb[r, :10]]
        GP["gp_iforest"][r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    if j % 200 == 0: print(f"  patient {j} | {time.time()-t0:.0f}s")
for k, v in GP.items(): df[k] = v
key = df.patient_id.astype(str) + "|" + df.anatom_site_general.astype(str)
df["gp_pos_k10_dist_siteadj"] = df["gp_pos_k10_dist"] / (df.groupby(key)["gp_pos_k10_dist"].transform("median") + 1e-6)
PRIOR = list(GP.keys()) + ["gp_pos_k10_dist_siteadj"]
print(f"graph prior: {len(PRIOR)} features in {time.time()-t0:.0f}s")

# zero-label score: average percentile rank of the 3 strongest label-free scores
U = np.nanmean(np.stack([pd.Series(df[c]).rank(pct=True).values for c in ["gp_tab_k10_lof", "gp_mahal", "gp_iforest"]]), 0)
U = np.nan_to_num(U, nan=0.5)
print("zero-label U score: AUROC", round(roc_auc_score(y, U), 4), "| pAUC80", round(pauc_80(y, U), 4))

In [ ]:
BASE_F = LESION_NUM + CAT_COLS
FSETS = {"M0_lesion": BASE_F, "M1_global": BASE_F + GLOBAL,
         "M2_prior": BASE_F + PRIOR, "M3_global_prior": BASE_F + GLOBAL + PRIOR}
MODELS = list(FSETS)
XALL = {m: df[f] for m, f in FSETS.items()}
BASE_PARAMS = dict(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7,
                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                   n_estimators=300, verbosity=-1)

def fit_predict(model, tr_rows, te_rows, seed):
    pos, neg = tr_rows[y[tr_rows] == 1], tr_rows[y[tr_rows] == 0]
    Xte = XALL[model].iloc[te_rows]; pred = np.zeros(len(te_rows))
    for b in range(N_BAGS):
        rng = np.random.default_rng(seed + b)
        sub = np.r_[pos, rng.choice(neg, min(len(neg), NEG_PER_POS * len(pos)), replace=False)]
        mcs = int(np.clip(len(sub) / 50, 5, 40))
        m = lgb.LGBMClassifier(**BASE_PARAMS, min_child_samples=mcs, random_state=seed + b).fit(XALL[model].iloc[sub], y[sub])
        pred += m.predict_proba(Xte)[:, 1] / N_BAGS
    return pred

pat = df.groupby("patient_id").agg(fold=("fold", "first"), has_pos=("target", "max"),
                                   hospital=("hospital", lambda s: s.mode().iat[0]))
rows_of = pd.Series(np.arange(n)).groupby(pid).indices

def sample_patients(ids_df, share, rng):
    chosen = []
    for hp in [0, 1]:
        ids = ids_df.index[ids_df.has_pos == hp].values
        k = int(round(share * len(ids)))
        if hp == 1: k = max(1, k)                      # at least one cancer patient
        if k > 0: chosen += list(rng.choice(ids, min(k, len(ids)), replace=False))
    return chosen

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj

In [ ]:
# extra detector for the ablation: Euclidean distance to the patient's centroid in shape/colour space
cent = pd.DataFrame(TAB).groupby(pid).transform("mean").values
df["gp_tab_centroid"] = np.linalg.norm(TAB - cent, axis=1)
codes = pd.factorize(pid)[0]; P = codes.max() + 1
print("patients:", P, "| U reproduced, AUROC =", round(roc_auc_score(y, U), 4))

## Fast patient-weighted metrics
A patient bootstrap is equivalent to giving every lesion the weight *(number of times its patient was drawn)*. Sorting once per score vector and using cumulative weights makes each resample O(n), which allows B = 1000.

In [ ]:
_trapz = getattr(np, "trapezoid", None) or getattr(np, "trapz")   # numpy 1.x and 2.x
def prep(yv, s, cv):
    o = np.argsort(s, kind="mergesort"); ss = s[o]
    return (yv[o], np.r_[np.diff(ss) != 0, True], cv[o])
def _roc(pr, cnt=None):
    ys, keep, co = pr; w = np.ones(len(ys)) if cnt is None else cnt[co].astype(float)
    tps = np.r_[0, np.cumsum(w * (ys == 0))[keep]]; fps = np.r_[0, np.cumsum(w * (ys == 1))[keep]]
    return fps / fps[-1], tps / tps[-1]
def pauc_p(pr, cnt=None, min_tpr=0.8):
    fpr, tpr = _roc(pr, cnt); mf = 1 - min_tpr; stop = np.searchsorted(fpr, mf, "right")
    tp = np.append(tpr[:stop], np.interp(mf, fpr[stop-1:stop+1], tpr[stop-1:stop+1])); fp = np.append(fpr[:stop], mf)
    return _trapz(tp, fp)
def auroc_p(pr, cnt=None):
    fpr, tpr = _roc(pr, cnt); return _trapz(tpr, fpr)
def ap_p(pr, cnt=None):
    ys, _, co = pr; ys = ys[::-1]; w = np.ones(len(ys)) if cnt is None else cnt[co[::-1]].astype(float)
    tp = np.cumsum(w * ys); prec = tp / np.maximum(np.cumsum(w), 1e-12)
    return float((prec * w * ys).sum() / max((w * ys).sum(), 1e-12))
_dummy = np.zeros(len(y), int)
def pauc_w(yv, s):  return pauc_p(prep(yv, s, _dummy[:len(yv)]))
def auroc_w(yv, s): return auroc_p(prep(yv, s, _dummy[:len(yv)]))
def ap_w(yv, s):    return ap_p(prep(yv, s, _dummy[:len(yv)]))

rng_b = np.random.default_rng(SEED)
BOOT_C = [np.bincount(rng_b.integers(0, P, P), minlength=P) for _ in range(N_BOOT)]
def boot_dist(metric_p, yv, scores, cv=None, n_boot=None):
    cv = codes if cv is None else cv; preps = [prep(yv, s, cv) for s in scores]
    C = BOOT_C if n_boot is None else BOOT_C[:n_boot]
    return np.array([np.mean([metric_p(pr, c) for pr in preps]) for c in C])
def ci(a): return f"[{np.percentile(a, 2.5):+.4f}, {np.percentile(a, 97.5):+.4f}]"

_s = np.random.default_rng(0).random(len(y)) + y
print("check pAUC:", round(pauc_80(y, _s), 6), round(pauc_w(y, _s), 6), "| AUROC:", round(roc_auc_score(y, _s), 6), round(auroc_w(y, _s), 6))

## Load saved predictions (Steps 4c, 5b, 5c)

In [ ]:
o4 = pd.read_csv(paths["oof_step4c.csv"]); o3 = pd.read_csv(paths["oof_step3.csv"])
assert (o4.isic_id.values == df.isic_id.values).all() and (o3.isic_id.values == df.isic_id.values).all()
L5B = dict(np.load(paths["step5b_lowlabel.npz"])); C1 = dict(np.load(paths["step5c_c1.npz"])); C2 = dict(np.load(paths["step5c_c2.npz"]))
HOSP = df.hospital.values
hosp_pos = df.groupby("hospital").target.sum(); HOSPS = sorted(hosp_pos[hosp_pos >= MIN_POS_HOSPITAL].index)

def rank_in(a, groups): return pd.Series(a).groupby(groups).rank(pct=True).values
rU_fold = rank_in(U, folds)
def blend(Uv, Sv, groups, w): return w * rank_in(Uv, groups) + (1 - w) * rank_in(Sv, groups)
def c1(q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    m3 = C1[f"M3|{q}|{r}"]; npos = C1[f"npos|{q}|{r}"]; w = (k / (k + npos))[folds]
    return dict(M3=m3, U=Uv, adaptive=w * rank_in(Uv, folds) + (1 - w) * rank_in(m3, folds))
def c2(h, q, r, k=6, Uv=None):
    Uv = U if Uv is None else Uv
    te = np.where(HOSP == h)[0]; m3 = C2[f"M3|{h}|{q}|{r}"]; n = C2[f"npos|{h}|{q}|{r}"][0]; w = k / (k + n)
    ru = pd.Series(Uv[te]).rank(pct=True).values; rm = pd.Series(m3).rank(pct=True).values
    return te, dict(M3=m3, U=Uv[te], adaptive=w * ru + (1 - w) * rm)
def c2_eval(q, k=6, Uv=None):
    out = []
    for h in HOSPS:
        acc = {m: [] for m in ["M3", "U", "adaptive"]}
        for r in range(REP_C2):
            te, sc = c2(h, q, r, k, Uv)
            for m in acc: acc[m].append(pauc_w(y[te], sc[m]))
        out.append({m: np.mean(v) for m, v in acc.items()})
    return pd.DataFrame(out, index=HOSPS)
print("loaded")

## R1 — Cohort table per centre

In [ ]:
g = df.groupby("hospital")
pp = df.groupby(["hospital", "patient_id"]).target.sum().reset_index()
lpp = df.groupby(["hospital", "patient_id"]).size().reset_index(name="n")
R1 = pd.DataFrame({
    "patients": g.patient_id.nunique(), "lesions": g.size(), "malignant": g.target.sum()})
R1["benign"] = R1.lesions - R1.malignant
R1["prevalence_%"] = (100 * R1.malignant / R1.lesions).round(3)
R1["patients_with_cancer"] = pp.groupby("hospital").target.apply(lambda s: int((s > 0).sum()))
R1["max_cancers_per_patient"] = pp.groupby("hospital").target.max()
R1["lesions_per_patient_median"] = lpp.groupby("hospital").n.median()
R1["lesions_per_patient_IQR"] = lpp.groupby("hospital").n.apply(lambda s: f"{int(s.quantile(.25))}-{int(s.quantile(.75))}")
tot = pd.DataFrame({"patients": [df.patient_id.nunique()], "lesions": [len(df)], "malignant": [int(y.sum())]}, index=["TOTAL"])
tot["benign"] = tot.lesions - tot.malignant; tot["prevalence_%"] = round(100 * y.mean(), 3)
tot["patients_with_cancer"] = int((pp.target > 0).sum()); tot["max_cancers_per_patient"] = int(pp.target.max())
lp = df.groupby("patient_id").size(); tot["lesions_per_patient_median"] = lp.median(); tot["lesions_per_patient_IQR"] = f"{int(lp.quantile(.25))}-{int(lp.quantile(.75))}"
R1 = pd.concat([R1.sort_values("malignant", ascending=False), tot])
print(R1.to_string())
print("\nCancers per positive patient (all centres):"); print(pp[pp.target > 0].target.value_counts().sort_index().to_string())
R1.to_csv(OUT("R1_cohort.csv"))

## R2 — Canonical numbers (point estimate + patient-bootstrap CI)
Point estimate = mean over repeats of the metric on the full evaluation set. Δ = difference of those point estimates. CI = percentile interval of the per-resample Δ (averaged over repeats). This replaces the earlier mix of bootstrap means and point estimates.

In [ ]:
t0 = time.time(); rows = []
# --- Step 5b: M3 vs M1, M2 vs M1 (10 repeats)
for q in FRAC_5B:
    S = {m: [L5B[f"{m}|{q}|{r}"] for r in range(REP_5B)] for m in ["M0_lesion", "M1_global", "M2_prior", "M3_global_prior"]}
    B = {m: boot_dist(pauc_p, y, v, n_boot=N_BOOT_REP) for m, v in S.items()}
    pt = {m: np.mean([pauc_w(y, s) for s in v]) for m, v in S.items()}
    for a, b in [("M3_global_prior", "M1_global"), ("M2_prior", "M1_global"), ("M3_global_prior", "M2_prior"), ("M2_prior", "M0_lesion")]:
        d = B[a] - B[b]
        rows.append(dict(exp="5b", budget=q, comparison=f"{a} vs {b}", a=round(pt[a], 4), b=round(pt[b], 4),
                         delta=round(pt[a] - pt[b], 4), ci95=ci(d), boot_p_le0=round((d <= 0).mean(), 3)))
# --- Step 5c C1: adaptive vs M3 / U (10 repeats)
for q in FRAC_C1:
    sc = [c1(q, r) for r in range(REP_C1)]
    B = {m: boot_dist(pauc_p, y, [s[m] for s in sc], n_boot=N_BOOT_REP) for m in ["M3", "U", "adaptive"]}
    pt = {m: np.mean([pauc_w(y, s[m]) for s in sc]) for m in ["M3", "U", "adaptive"]}
    for b in ["M3", "U"]:
        d = B["adaptive"] - B[b]
        rows.append(dict(exp="C1", budget=q, comparison=f"adaptive vs {b}", a=round(pt["adaptive"], 4), b=round(pt[b], 4),
                         delta=round(pt["adaptive"] - pt[b], 4), ci95=ci(d), boot_p_le0=round((d <= 0).mean(), 3)))
# --- full data: G3/G4 vs G1
for a, b in [("pgb2_G4_pgboost_v2", "pgb2_G1_global"), ("pgb2_G3_pgboost_v1", "pgb2_G1_global"), ("stack_v1", "stack_global")]:
    d = boot_dist(pauc_p, y, [o4[a].values]) - boot_dist(pauc_p, y, [o4[b].values])
    rows.append(dict(exp="full", budget=1.0, comparison=f"{a} vs {b}", a=round(pauc_w(y, o4[a].values), 4), b=round(pauc_w(y, o4[b].values), 4),
                     delta=round(pauc_w(y, o4[a].values) - pauc_w(y, o4[b].values), 4), ci95=ci(d), boot_p_le0=round((d <= 0).mean(), 3)))
R2 = pd.DataFrame(rows); print(R2.to_string(index=False)); print(f"({(time.time()-t0)/60:.1f} min)")
R2.to_csv(OUT("R2_canonical.csv"), index=False)

# --- C2: both aggregation rules side by side
rows = []
for q in FRAC_C2:
    per = c2_eval(q)
    rows.append(dict(budget=q, macro_M3=per.M3.mean(), macro_U=per.U.mean(), macro_adaptive=per.adaptive.mean(),
                     macro_delta_vs_M3=per.adaptive.mean() - per.M3.mean(), macro_delta_vs_U=per.adaptive.mean() - per.U.mean(),
                     centres_better_than_M3=int((per.adaptive > per.M3).sum()), centres_better_than_U=int((per.adaptive > per.U).sum())))
R2c2 = pd.DataFrame(rows).round(4); print(R2c2.to_string(index=False)); R2c2.to_csv(OUT("R2_C2_macro.csv"), index=False)
print("Note: the random-effects pooled estimates in tests_step5c_c2_meta.csv weight centres by inverse variance, so they differ from these unweighted macro means.")

## R3 — Sensitivity to *k* (no selection on the evaluation data)

In [ ]:
M3_C1 = {q: np.mean([pauc_w(y, C1[f"M3|{q}|{r}"]) for r in range(REP_C1)]) for q in FRAC_C1}; U_PAUC = pauc_w(y, U)
rows = []
for k in K_GRID:
    for q in FRAC_C1:
        v = np.mean([pauc_w(y, c1(q, r, k)["adaptive"]) for r in range(REP_C1)])
        m3 = M3_C1[q]; u = U_PAUC
        rows.append(dict(setting="C1", k=k, budget=q, adaptive=v, best_of=max(m3, u), regret=v - max(m3, u)))
    for q in FRAC_C2:
        per = c2_eval(q, k)
        rows.append(dict(setting="C2", k=k, budget=q, adaptive=per.adaptive.mean(), best_of=max(per.M3.mean(), per.U.mean()),
                         regret=per.adaptive.mean() - max(per.M3.mean(), per.U.mean())))
R3 = pd.DataFrame(rows)
summ = R3.groupby(["setting", "k"]).regret.agg(mean_regret="mean", worst_regret="min").round(4).unstack(0)
print("Regret = adaptive - best of (M3, U); positive is better. Worst = most negative budget.")
print(summ.to_string())
print(R3.pivot_table(index=["setting", "budget"], columns="k", values="adaptive").round(4).to_string())
R3.to_csv(OUT("R3_k_sensitivity.csv"), index=False)

## R4 — Ablation of the zero-label score

In [ ]:
def prank(c): return pd.Series(df[c].values).rank(pct=True).fillna(0.5).values
DET = {"centroid distance": prank("gp_tab_centroid"), "kNN distance (k=10)": prank("gp_tab_k10_dist"),
       "LOF (k=10)": prank("gp_tab_k10_lof"), "Mahalanobis": prank("gp_mahal"), "Isolation forest": prank("gp_iforest")}
DET["LOF + Mahalanobis"] = (DET["LOF (k=10)"] + DET["Mahalanobis"]) / 2
DET["U = LOF + Mahal. + IF"] = U
DET["U + appearance LOF"] = (3 * U + prank("gp_app_k10_lof")) / 4
def auc_within(s):
    out = []
    for _, d in pd.DataFrame({"y": y, "s": s, "p": codes}).groupby("p"):
        if d.y.nunique() == 2: out.append((roc_auc_score(d.y, d.s), d.y.sum()))
    a = np.array(out); return (a[:, 0] * a[:, 1]).sum() / a[:, 1].sum()
rows = []; t0 = time.time()
for name, s in DET.items():
    bp, ba = boot_dist(pauc_p, y, [s]), boot_dist(auroc_p, y, [s])
    blend_c1 = {f"adaptive@{int(q*100)}%": np.mean([pauc_w(y, c1(q, r, 6, s)["adaptive"]) for r in range(REP_C1)]) for q in [0.01, 0.02, 0.05, 0.10]}
    rows.append(dict(detector=name, pAUC80=pauc_w(y, s), pAUC_ci=ci(bp), AUROC=auroc_w(y, s), AUROC_ci=ci(ba),
                     AUROC_within_patient=auc_within(s), AP=ap_w(y, s), **blend_c1))
R4 = pd.DataFrame(rows).round(4); print(R4.to_string(index=False)); print(f"({(time.time()-t0)/60:.1f} min)")
R4.to_csv(OUT("R4_U_ablation.csv"), index=False)

## R5 — Clinical metrics

In [ ]:
def topk(s, k):
    r = pd.Series(s).groupby(codes).rank(ascending=False, method="first").values
    f = r <= k; tp = (f & (y == 1)).sum()
    return dict(recall=tp / y.sum(), precision=tp / f.sum(), fp_per_patient=(f & (y == 0)).sum() / P, flagged=f.sum())
def workload(s, frac):
    thr = np.quantile(s, 1 - frac); f = s >= thr
    return (f & (y == 1)).sum() / y.sum(), f.sum() / P
def recall_k_boot(s, k):
    r = pd.Series(s).groupby(codes).rank(ascending=False, method="first").values
    hits = np.bincount(codes, weights=((r <= k) & (y == 1)).astype(float), minlength=P); pos = np.bincount(codes, weights=y.astype(float), minlength=P)
    rb = np.random.default_rng(SEED); out = []
    for _ in range(N_BOOT):
        c = np.bincount(rb.integers(0, P, P), minlength=P); out.append((c * hits).sum() / max((c * pos).sum(), 1))
    return np.array(out)
def clin_row(name, scores):
    d = dict(model=name, AP=np.mean([ap_w(y, s) for s in scores]))
    for k in [5, 10]:
        m = [topk(s, k) for s in scores]
        d[f"recall@{k}"] = np.mean([x["recall"] for x in m]); d[f"precision@{k}"] = np.mean([x["precision"] for x in m]); d[f"FP/patient@{k}"] = np.mean([x["fp_per_patient"] for x in m])
    rb = np.mean([recall_k_boot(s, 5) for s in scores], axis=0) if len(scores) == 1 else None
    d["recall@5_ci"] = ci(rb) if rb is not None else ""
    for fr in [0.01, 0.02, 0.05]:
        v = [workload(s, fr) for s in scores]; d[f"sens@refer{int(fr*100)}%"] = np.mean([a for a, _ in v]); d[f"referred/patient@{int(fr*100)}%"] = np.mean([b for _, b in v])
    return d
rows = [clin_row(n, [v]) for n, v in [("tab_lesion", o3.oof_tab_lesion.values), ("tab_pnorm", o3.oof_tab_pnorm.values),
                                      ("G1 global", o4.pgb2_G1_global.values), ("G4 + prior", o4.pgb2_G4_pgboost_v2.values),
                                      ("G3+img (best)", o4.stack_v1.values), ("U (zero-label)", U)]]
for q in [0.01, 0.02, 0.05, 0.10]:
    sc = [c1(q, r) for r in range(REP_C1)]
    for m in ["M3", "adaptive"]:
        rows.append(clin_row(f"C1 {int(q*100)}% {m}", [s[m] for s in sc]))
R5 = pd.DataFrame(rows).round(4); pd.set_option("display.width", 250)
print(R5.to_string(index=False)); R5.to_csv(OUT("R5_clinical_metrics.csv"), index=False)
print(f"\nBaseline: AP of a random ranking = prevalence = {y.mean():.5f}")

## R6 — Paired patient-level permutation tests (prespecified primary hypotheses, Holm)
Under H0 the two models are exchangeable within each patient: we swap the two models' (rank-normalised) scores for a random half of the patients, recompute Δ (averaged over repeats), and compare with the observed Δ. One-sided p = (1 + #{Δ_perm ≥ Δ_obs}) / (1 + N_PERM).

In [ ]:
def perm_test(a_list, b_list, n_perm=N_PERM, seed=SEED):
    A = [pd.Series(a).rank(pct=True).values.astype(np.float32) for a in a_list]; B = [pd.Series(b).rank(pct=True).values.astype(np.float32) for b in b_list]
    obs = np.mean([pauc_w(y, a) - pauc_w(y, b) for a, b in zip(A, B)])
    rng = np.random.default_rng(seed); ge = 0
    for _ in range(n_perm):
        sw = (rng.random(P) < 0.5)[codes]
        d = np.mean([pauc_w(y, np.where(sw, b, a)) - pauc_w(y, np.where(sw, a, b)) for a, b in zip(A, B)])
        ge += d >= obs
    return obs, (1 + ge) / (1 + n_perm)
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.zeros(len(p)); run = 0
    for rank, i in enumerate(o): run = max(run, min(1.0, (len(p) - rank) * p[i])); adj[i] = run
    return adj
t0 = time.time(); fams = []
fam = []
for q in FRAC_5B:
    a = [L5B[f"M3_global_prior|{q}|{r}"] for r in range(REP_5B)]; b = [L5B[f"M1_global|{q}|{r}"] for r in range(REP_5B)]
    d, p = perm_test(a, b); fam.append(dict(family="label-efficiency", test=f"M3 vs M1 @{int(q*100)}%", delta=d, p_perm=p))
fams.append(fam); fam = []
for q in [0.01, 0.02, 0.03]:
    sc = [c1(q, r) for r in range(REP_C1)]
    d, p = perm_test([s["adaptive"] for s in sc], [s["M3"] for s in sc]); fam.append(dict(family="adaptive vs M3", test=f"C1 @{int(q*100)}%", delta=d, p_perm=p))
fams.append(fam); fam = []
for q in [0.01, 0.02, 0.05, 0.10]:
    sc = [c1(q, r) for r in range(REP_C1)]
    d, p = perm_test([s["adaptive"] for s in sc], [s["U"] for s in sc]); fam.append(dict(family="adaptive vs U", test=f"C1 @{int(q*100)}%", delta=d, p_perm=p))
fams.append(fam); fam = []
for a, b in [("pgb2_G4_pgboost_v2", "pgb2_G1_global"), ("pgb2_G3_pgboost_v1", "pgb2_G1_global")]:
    d, p = perm_test([o4[a].values], [o4[b].values]); fam.append(dict(family="full data", test=f"{a} vs {b}", delta=d, p_perm=p))
fams.append(fam)
R6 = []
for fam in fams:
    adj = holm([f["p_perm"] for f in fam])
    for f, a in zip(fam, adj): f["p_holm"] = a; R6.append(f)
R6 = pd.DataFrame(R6).round(4); print(R6.to_string(index=False)); print(f"resolution = 1/{N_PERM+1};  ({(time.time()-t0)/60:.1f} min)")
R6.to_csv(OUT("R6_permutation_tests.csv"), index=False)

## R7 — Deployment-style ranking
(a) **Within-patient ranks**: both scores ranked inside each patient (per-patient triage list) — evaluated with per-patient top-k metrics.
(b) **Reference-ECDF ranks**: each new lesion is ranked against score distributions from an *unlabelled reference pool* (training-side patients that were not labelled). This scores a single patient without any test batch. Requires re-training M3 (new seeds), so both the batch version and the reference version are computed from the same models.

In [ ]:
rows = []
for q in [0.01, 0.02, 0.05, 0.10]:
    for r in range(REP_C1):
        s = c1(q, r); w = (6 / (6 + C1[f"npos|{q}|{r}"]))[folds]
        per_pat = w * rank_in(U, codes) + (1 - w) * rank_in(s["M3"], codes)
        for name, v in [("batch rank (paper)", s["adaptive"]), ("within-patient rank", per_pat), ("M3", s["M3"]), ("U", U)]:
            m5, m10 = topk(v, 5), topk(v, 10)
            rows.append(dict(budget=q, rep=r, method=name, recall5=m5["recall"], precision5=m5["precision"], recall10=m10["recall"]))
R7a = pd.DataFrame(rows).groupby(["budget", "method"])[["recall5", "precision5", "recall10"]].mean().round(4)
print(R7a.to_string()); R7a.to_csv(OUT("R7a_within_patient.csv"))

In [ ]:
if RUN_REFERENCE:
    t0 = time.time(); rows = []; REF_N = 20000; REP_R = 5
    for q in [0.01, 0.02, 0.05, 0.10]:
        for r in range(REP_R):
            batch = np.zeros(n); refv = np.zeros(n); m3all = np.zeros(n); npos = np.zeros(N_FOLDS)
            for f in range(N_FOLDS):
                rng = np.random.default_rng(SEED_C + 555 + 7919 * r + 101 * f + int(q * 10000))
                cand = pat[pat.fold != f]; chosen = sample_patients(cand, q, rng)
                tr = np.concatenate([rows_of[p] for p in chosen]); te = np.where(folds == f)[0]
                pool_p = [p for p in cand.index if p not in set(chosen)]
                pool = np.concatenate([rows_of[p] for p in pool_p]); pool = rng.choice(pool, min(REF_N, len(pool)), replace=False)
                both = np.r_[te, pool]
                pr = fit_predict("M3_global_prior", tr, both, SEED_C + 13 * f + 1000 * r)
                s_te, s_pool = pr[:len(te)], pr[len(te):]
                n_c = y[tr].sum(); npos[f] = n_c; w = 6 / (6 + n_c)
                ecdf = lambda v, ref: np.searchsorted(np.sort(ref), v, side="right") / len(ref)
                refv[te] = w * ecdf(U[te], U[pool]) + (1 - w) * ecdf(s_te, s_pool)
                m3all[te] = s_te
            wv = (6 / (6 + npos))[folds]
            batch = wv * rank_in(U, folds) + (1 - wv) * rank_in(m3all, folds)
            rows.append(dict(budget=q, rep=r, M3=pauc_w(y, m3all), batch_rank=pauc_w(y, batch), reference_ecdf=pauc_w(y, refv),
                             recall5_batch=topk(batch, 5)["recall"], recall5_reference=topk(refv, 5)["recall"]))
        print(f"q={q}: done ({(time.time()-t0)/60:.1f} min)")
    R7b = pd.DataFrame(rows).groupby("budget").mean().drop(columns="rep").round(4)
    print(R7b.to_string()); R7b.to_csv(OUT("R7b_reference_ecdf.csv"))

## R8 — Subgroups by number of lesions per patient

In [ ]:
npl = pd.Series(codes).map(pd.Series(codes).value_counts()).values
bins = np.quantile(pd.Series(codes).value_counts().values, [0, 1/3, 2/3, 1])
grp = np.digitize(npl, bins[1:-1], right=True)
labels = [f"<= {int(bins[1])}", f"{int(bins[1])+1}-{int(bins[2])}", f"> {int(bins[2])}"]
rows = []
cands = {"tab_pnorm (full)": [o3.oof_tab_pnorm.values], "U": [U]}
for q in [0.02, 0.10]:
    sc = [c1(q, r) for r in range(REP_C1)]
    cands[f"M3 C1 {int(q*100)}%"] = [s["M3"] for s in sc]; cands[f"adaptive C1 {int(q*100)}%"] = [s["adaptive"] for s in sc]
for gi, lab in enumerate(labels):
    m = grp == gi
    row = dict(lesions_per_patient=lab, patients=len(np.unique(codes[m])), cancers=int(y[m].sum()))
    for name, sc in cands.items():
        row[f"AUROC {name}"] = np.mean([auroc_w(y[m], s[m]) for s in sc])
    rows.append(row)
R8 = pd.DataFrame(rows).round(4); print(R8.to_string(index=False)); R8.to_csv(OUT("R8_lesion_count_subgroups.csv"), index=False)

## R9 — Leakage audit
**(a) Nested vs flat stacking.** Base models: tabular LightGBM (lesion features) and DINOv2 logistic regression. *Flat* = the paper's procedure (meta-learner trained on OOF predictions whose base models saw the test fold). *Nested* = for each outer fold, inner OOF predictions from base models that never saw the outer test fold. The CNN is not re-trained (GPU), so this audit uses the two CPU base models.
**(b) Zero-label score with preprocessing fitted on training patients only** (scaler + PCA fitted on the other four folds, then applied to the test fold).

In [ ]:
if RUN_NESTED:
    t0 = time.time()
    EMB = np.load(paths["emb_dinov2_f16.npy"]).astype(np.float32)
    FT = LESION_NUM + CAT_COLS; Xtab = df[FT]
    def fit_tab(tr, te, seed):
        pos, neg = tr[y[tr] == 1], tr[y[tr] == 0]; out = np.zeros(len(te))
        for b in range(N_BAGS):
            rg = np.random.default_rng(seed + b); sub = np.r_[pos, rg.choice(neg, int(0.01 * len(neg)), replace=False)]
            m = lgb.LGBMClassifier(objective="binary", learning_rate=0.03, num_leaves=63, max_depth=7, min_child_samples=40,
                                   feature_fraction=0.6, bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=1.0,
                                   n_estimators=300, verbosity=-1, random_state=seed + b).fit(Xtab.iloc[sub], y[sub])
            out += m.predict_proba(Xtab.iloc[te])[:, 1] / N_BAGS
        return out
    def fit_emb(tr, te, seed):
        rg = np.random.default_rng(seed); pos = tr[y[tr] == 1]; neg = rg.choice(tr[y[tr] == 0], min(50 * len(pos), (y[tr] == 0).sum()), replace=False)
        sub = np.r_[pos, neg]; sc = StandardScaler().fit(EMB[sub])
        return LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced").fit(sc.transform(EMB[sub]), y[sub]).predict_proba(sc.transform(EMB[te]))[:, 1]
    logit = lambda p: np.log(np.clip(p, 1e-6, 1 - 1e-6) / (1 - np.clip(p, 1e-6, 1 - 1e-6)))
    FOLD = [np.where(folds == f)[0] for f in range(N_FOLDS)]
    outer = {b: np.zeros(n) for b in ["tab", "emb"]}
    for f in range(N_FOLDS):
        tr = np.where(folds != f)[0]
        outer["tab"][FOLD[f]] = fit_tab(tr, FOLD[f], SEED + f); outer["emb"][FOLD[f]] = fit_emb(tr, FOLD[f], SEED + f)
    Zflat = np.c_[logit(outer["tab"]), logit(outer["emb"])]
    flat, nested = np.zeros(n), np.zeros(n)
    for f in range(N_FOLDS):
        tr = np.where(folds != f)[0]
        sc = StandardScaler().fit(Zflat[tr]); m = LogisticRegression(max_iter=2000).fit(sc.transform(Zflat[tr]), y[tr])
        flat[FOLD[f]] = m.predict_proba(sc.transform(Zflat[FOLD[f]]))[:, 1]
        inner = np.zeros((n, 2))
        for g in [g for g in range(N_FOLDS) if g != f]:
            itr = np.where((folds != f) & (folds != g))[0]
            inner[FOLD[g], 0] = logit(fit_tab(itr, FOLD[g], SEED + 100 + 10 * f + g)); inner[FOLD[g], 1] = logit(fit_emb(itr, FOLD[g], SEED + 100 + 10 * f + g))
        sc = StandardScaler().fit(inner[tr]); m = LogisticRegression(max_iter=2000).fit(sc.transform(inner[tr]), y[tr])
        nested[FOLD[f]] = m.predict_proba(sc.transform(Zflat[FOLD[f]]))[:, 1]
        print(f"  outer fold {f} done ({(time.time()-t0)/60:.1f} min)")
    d = boot_dist(pauc_p, y, [flat]) - boot_dist(pauc_p, y, [nested])
    R9a = pd.DataFrame([dict(base_tab=pauc_w(y, outer["tab"]), base_emb=pauc_w(y, outer["emb"]), flat_stack=pauc_w(y, flat),
                             nested_stack=pauc_w(y, nested), flat_minus_nested=pauc_w(y, flat) - pauc_w(y, nested), ci95=ci(d))]).round(4)
    print(R9a.to_string(index=False)); R9a.to_csv(OUT("R9a_nested_stacking.csv"), index=False)
    del EMB; gc.collect()

In [ ]:
# (b) zero-label score with scaler/PCA fitted on training-fold patients only
t0 = time.time(); U_fold = np.full(n, np.nan)
for f in range(N_FOLDS):
    trm, te = folds != f, np.where(folds == f)[0]
    scl = StandardScaler().fit(X[trm][:, tab_idx]); T = scl.transform(X[:, tab_idx]).clip(-6, 6).astype(np.float32)
    pca = PCA(10, random_state=SEED).fit(T[trm]); T10 = pca.transform(T).astype(np.float32)
    lofv, mah, ifo = np.full(n, np.nan), np.full(n, np.nan), np.full(n, np.nan)
    for p_, rows_ in pd.Series(te).groupby(pid[te]).indices.items():
        r = te[np.asarray(rows_)]; m = len(r)
        if m < 6: continue
        km = min(20, m - 1); dist, idx = NearestNeighbors(n_neighbors=km + 1).fit(T[r]).kneighbors(T[r])
        lofv[r] = lof(dist[:, 1:], idx[:, 1:], min(10, km))
        if m >= 15:
            Zp = T10[r]; dz = Zp - Zp.mean(0); C = np.cov(Zp, rowvar=False) + 1e-3 * np.eye(10)
            mah[r] = np.sqrt(np.einsum("ij,jk,ik->i", dz, np.linalg.inv(C), dz))
        if m >= 10:
            Zi = np.c_[T10[r], emb[r, :10]]; ifo[r] = -IsolationForest(n_estimators=100, random_state=SEED).fit(Zi).score_samples(Zi)
    for a in (lofv, mah, ifo): pass
    U_fold[te] = np.nanmean(np.stack([pd.Series(v[te]).rank(pct=True).values for v in (lofv, mah, ifo)]), 0)
U_fold = np.nan_to_num(U_fold, nan=0.5)
R9b = pd.DataFrame([dict(version="U (paper: preprocessing fitted on all lesions, label-free)", pAUC80=pauc_w(y, rank_in(U, folds)), AUROC=auroc_w(y, U)),
                    dict(version="U (scaler/PCA fitted on training folds only)", pAUC80=pauc_w(y, U_fold), AUROC=auroc_w(y, U_fold))]).round(4)
print(R9b.to_string(index=False)); print(f"({(time.time()-t0)/60:.1f} min)"); R9b.to_csv(OUT("R9b_U_foldwise_preprocessing.csv"), index=False)
print("Note: the DINOv2 PCA-32 (appearance) used inside the isolation forest is unsupervised and fitted once; no labels enter any of these steps.")

## Bundle the revision results

In [ ]:
import zipfile
from IPython.display import FileLink
with zipfile.ZipFile(f"{WORK}/revision_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{WORK}/rev_*.csv")): z.write(f, os.path.basename(f))
print(sorted(os.path.basename(f) for f in glob.glob(f"{WORK}/rev_*.csv")))
FileLink("revision_results.zip")